#Part B — Analysis

In [ ]:

fg['date'] = pd.to_datetime(fg['date'])
hd['date'] = pd.to_datetime(hd['Timestamp IST'], format='%d-%m-%Y %H:%M').dt.date
hd['date'] = pd.to_datetime(hd['date'])

merged = hd.merge(fg[['date', 'value', 'classification']], on='date', how='left')
merged['is_win']   = merged['Closed PnL'] > 0
merged['is_long']  = merged['Direction'].str.contains('Long',  na=False)
merged['is_short'] = merged['Direction'].str.contains('Short', na=False)

# broad sentiment bucket
merged['sentiment'] = merged['classification'].map({
    'Extreme Fear': 'Fear', 'Fear': 'Fear',
    'Neutral':      'Neutral',
    'Greed':        'Greed', 'Extreme Greed': 'Greed'
})

# leverage proxy
merged['leverage'] = merged['Size USD'] / merged.groupby('Account')['Size USD'].transform('mean')



# Q1 — PnL, Win Rate, Drawdown Proxy by Sentiment


# daily total PnL per sentiment
daily = merged.groupby(['date', 'sentiment']).agg(
    total_pnl   = ('Closed PnL', 'sum'),
    win_rate    = ('is_win',     'mean'),
    num_trades  = ('Closed PnL', 'count')
).reset_index()

# drawdown proxy = negative PnL days average
drawdown = daily[daily['total_pnl'] < 0].groupby('sentiment')['total_pnl'].mean()

q1_table = daily.groupby('sentiment').agg(
    avg_daily_pnl = ('total_pnl',  'mean'),
    avg_win_rate  = ('win_rate',   'mean'),
).round(2)
q1_table['avg_drawdown_proxy'] = drawdown.round(2)

print("Q1 — Performance by Sentiment")
print(q1_table)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Q1 — Performance by Sentiment', fontsize=13)
order = ['Fear', 'Neutral', 'Greed']
colors = ['#d62728', '#bcbd22', '#2ca02c']

axes[0].bar(order, q1_table.loc[order, 'avg_daily_pnl'], color=colors)
axes[0].set_title('Avg Daily PnL')
axes[0].set_ylabel('USD')

axes[1].bar(order, q1_table.loc[order, 'avg_win_rate'] * 100, color=colors)
axes[1].set_title('Avg Win Rate (%)')
axes[1].set_ylabel('%')

axes[2].bar(order, drawdown.loc[order].abs(), color=colors)
axes[2].set_title('Avg Drawdown (loss days)')
axes[2].set_ylabel('USD (abs)')

plt.tight_layout()
plt.savefig('q1_performance_by_sentiment.png', dpi=150)
plt.show()


# Q2 Trader Behavior by Sentiment


behavior = merged.groupby('sentiment').agg(
    avg_trades_per_day = ('Closed PnL',  'count'),
    avg_leverage       = ('leverage',    'mean'),
    avg_position_size  = ('Size USD',    'mean'),
    long_ratio         = ('is_long',     'mean'),
    short_ratio        = ('is_short',    'mean'),
).round(3)
behavior['long_short_ratio'] = (behavior['long_ratio'] / behavior['short_ratio']).round(2)

print("\nQ2 — Behavior by Sentiment")
print(behavior[['avg_trades_per_day', 'avg_leverage', 'avg_position_size', 'long_short_ratio']])

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Q2 — Trader Behavior by Sentiment', fontsize=13)
metrics = ['avg_trades_per_day', 'avg_leverage', 'avg_position_size', 'long_short_ratio']
titles  = ['Avg Trades', 'Avg Leverage', 'Avg Position Size (USD)', 'Long/Short Ratio']

for ax, col, title in zip(axes, metrics, titles):
    ax.bar(order, behavior.loc[order, col], color=colors)
    ax.set_title(title)
    ax.axhline(behavior[col].mean(), color='black', lw=1, ls='--', label='overall avg')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('q2_behavior_by_sentiment.png', dpi=150)
plt.show()



# Q3 Trader Segments


account_stats = merged.groupby('Account').agg(
    total_trades  = ('Closed PnL', 'count'),
    total_pnl     = ('Closed PnL', 'sum'),
    win_rate      = ('is_win',     'mean'),
    avg_leverage  = ('leverage',   'mean'),
    avg_size      = ('Size USD',   'mean'),
).reset_index()

# Segment 1 High vs Low Leverage
account_stats['leverage_segment'] = pd.qcut(
    account_stats['avg_leverage'], q=2, labels=['Low Leverage', 'High Leverage']
)

# Segment 2 Frequent vs Infrequent
account_stats['freq_segment'] = pd.qcut(
    account_stats['total_trades'], q=2, labels=['Infrequent', 'Frequent']
)

# Segment 3 Consistent Winners vs Inconsistent
account_stats['winner_segment'] = account_stats['win_rate'].apply(
    lambda x: 'Consistent Winner' if x >= 0.5 else 'Inconsistent'
)

print("\nQ3 — Segment Summaries")
for seg_col in ['leverage_segment', 'freq_segment', 'winner_segment']:
    print(f"\n{seg_col}:")
    print(account_stats.groupby(seg_col)[['total_pnl','win_rate','avg_leverage','avg_size']].mean().round(2))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Q3 — Trader Segments', fontsize=13)

# Segment 1
seg1 = account_stats.groupby('leverage_segment')['total_pnl'].mean()
axes[0].bar(seg1.index, seg1.values, color=['#1f77b4', '#ff7f0e'])
axes[0].set_title('Avg PnL: High vs Low Leverage')
axes[0].set_ylabel('Avg Total PnL (USD)')

# Segment 2
seg2 = account_stats.groupby('freq_segment')['win_rate'].mean() * 100
axes[1].bar(seg2.index, seg2.values, color=['#2ca02c', '#d62728'])
axes[1].set_title('Win Rate: Frequent vs Infrequent')
axes[1].set_ylabel('Win Rate (%)')

# Segment 3
seg3 = account_stats.groupby('winner_segment')['total_pnl'].mean()
axes[2].bar(seg3.index, seg3.values, color=['#17becf', '#e377c2'])
axes[2].set_title('Avg PnL: Winners vs Inconsistent')
axes[2].set_ylabel('Avg Total PnL (USD)')

plt.tight_layout()
plt.savefig('q3_segments.png', dpi=150)
plt.show()



# INSIGHTS SUMMARY

print("\n── Insight 1 ──")
fear_pnl  = q1_table.loc['Fear',  'avg_daily_pnl']
greed_pnl = q1_table.loc['Greed', 'avg_daily_pnl']
print(f"Fear days avg PnL: {fear_pnl:.2f} | Greed days avg PnL: {greed_pnl:.2f}")

print("\n── Insight 2 ──")
fear_ls  = behavior.loc['Fear',  'long_short_ratio']
greed_ls = behavior.loc['Greed', 'long_short_ratio']
print(f"Fear L/S ratio: {fear_ls} | Greed L/S ratio: {greed_ls}")

print("\n── Insight 3 ──")
high_lev_pnl = account_stats.groupby('leverage_segment')['total_pnl'].mean()
print(high_lev_pnl)